# 🔍 Multimodal RAG with ChromaDB + OpenAI
**Stack:** `pdf2image` · `pdfplumber` · `PyMuPDF` · `ChromaDB` · `OpenAI GPT-4o` · `text-embedding-3-small`

## What This Notebook Does
| Step | Action |
|------|--------|
| 1 | Upload a PDF (text + image pages) |
| 2 | Extract **text** per page with `pdfplumber` |
| 3 | Rasterize **every page** as a PNG image |
| 4 | Use **GPT-4o Vision** to describe image-heavy pages → rich text descriptions |
| 5 | Chunk & embed all content with `text-embedding-3-small` |
| 6 | Store chunks in **ChromaDB** with metadata (page, type, source) |
| 7 | At query time: retrieve top-k chunks → pass context + original page image to **GPT-4o** |
| 8 | Get a **grounded, cited answer** referencing specific pages |

> **Sample PDF to test with:** Use `TechNova_Q4_Report.pdf` (3-page doc with text, tables, and a dashboard image on Page 2)


## 📦 Cell 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install -q chromadb openai pdfplumber pymupdf pdf2image pillow                langchain langchain-openai langchain-community                langchain-text-splitters tiktoken

# Install poppler (needed by pdf2image for page rasterization)
!apt-get -qq install -y poppler-utils

print("✅ All packages installed")


## 🔑 Cell 2 — Imports & OpenAI API Key

In [ ]:
import os
import base64
import io
import json
import textwrap
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import fitz                          # PyMuPDF — page rasterization
import pdfplumber                    # text extraction
from pdf2image import convert_from_path
from PIL import Image
import chromadb
from chromadb.utils import embedding_functions
import openai
from openai import OpenAI

# ── Set your OpenAI API key ──────────────────────────────────────
# Option A: Type it directly (fine for personal Colab)
OPENAI_API_KEY = "sk-..."           # ← replace with your key

# Option B: Load from Colab Secrets (recommended)
# from google.colab import userdata
# OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)

print("✅ OpenAI client ready")


## 📄 Cell 3 — Upload Your PDF

In [ ]:
from google.colab import files

print("Upload your PDF file (e.g. TechNova_Q4_Report.pdf)...")
uploaded = files.upload()

PDF_PATH = list(uploaded.keys())[0]
print(f"✅ Uploaded: {PDF_PATH}")

# Quick check
doc_check = fitz.open(PDF_PATH)
print(f"   Pages: {doc_check.page_count}")
doc_check.close()


## ⚙️ Cell 4 — Configuration

In [ ]:
# ── Tunable parameters ────────────────────────────────────────────
CHUNK_SIZE          = 600    # characters per text chunk
CHUNK_OVERLAP       = 80     # overlap between chunks for context continuity
TOP_K               = 4      # number of chunks to retrieve per query
IMAGE_DPI           = 150    # DPI for page rasterization (150 = good quality/speed balance)
EMBED_MODEL         = "text-embedding-3-small"
VISION_MODEL        = "gpt-4o"
GENERATION_MODEL    = "gpt-4o"

# Image description: only describe a page with vision if text extraction
# yields fewer than MIN_TEXT_CHARS characters (likely image-heavy page)
MIN_TEXT_CHARS      = 200

CHROMA_COLLECTION   = "multimodal_rag"
CHROMA_DIR          = "./chroma_db"

print("✅ Config set")
print(f"   Chunk size={CHUNK_SIZE}, overlap={CHUNK_OVERLAP}, top-k={TOP_K}")
print(f"   Embed: {EMBED_MODEL}  |  Vision/Generation: {VISION_MODEL}")


## 📝 Cell 5 — Extract Text from Each Page
Uses `pdfplumber` for accurate layout-aware text extraction.


In [ ]:
def extract_text_per_page(pdf_path: str) -> List[Dict]:
    """
    Returns a list of dicts, one per page:
    {
      'page_num': int (1-indexed),
      'text': str,
      'char_count': int
    }
    """
    pages = []
    with pdfplumber.open(pdf_path) as pdf:
        for i, page in enumerate(pdf.pages):
            text = page.extract_text() or ""
            # Also extract tables as text to preserve tabular data
            tables = page.extract_tables()
            if tables:
                for table in tables:
                    for row in table:
                        row_text = "  |  ".join(
                            cell.strip() if cell else "" for cell in row
                        )
                        text += "\n" + row_text
            pages.append({
                "page_num": i + 1,
                "text": text.strip(),
                "char_count": len(text.strip())
            })
    return pages


text_pages = extract_text_per_page(PDF_PATH)

print(f"✅ Text extracted from {len(text_pages)} pages")
for p in text_pages:
    print(f"   Page {p['page_num']}: {p['char_count']:,} chars — "
          f"{'[text-rich]' if p['char_count'] >= MIN_TEXT_CHARS else '[image-heavy ⇒ will use vision]'}")


## 🖼️ Cell 6 — Rasterize Pages as Images
Every page is converted to a PNG at `IMAGE_DPI` resolution.  
These images are used both for GPT-4o Vision descriptions and for
passing visual context when answering questions about image pages.


In [ ]:
def rasterize_pages(pdf_path: str, dpi: int = 150) -> List[Dict]:
    """
    Rasterize all PDF pages to PIL Images.
    Returns: [{'page_num': int, 'image': PIL.Image, 'base64': str}, ...]
    """
    pil_images = convert_from_path(pdf_path, dpi=dpi)
    results = []
    for i, img in enumerate(pil_images):
        # Encode to base64 for OpenAI vision API
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
        results.append({
            "page_num": i + 1,
            "image": img,
            "base64": b64,
            "size": img.size
        })
    return results


page_images = rasterize_pages(PDF_PATH, dpi=IMAGE_DPI)

print(f"✅ Rasterized {len(page_images)} pages")
for p in page_images:
    print(f"   Page {p['page_num']}: {p['size'][0]}×{p['size'][1]} px")

# Preview page 1
from IPython.display import display
display(page_images[0]['image'].resize(
    (int(page_images[0]['image'].width * 0.4),
     int(page_images[0]['image'].height * 0.4))
))
print("↑ Page 1 preview")


## 👁️ Cell 7 — GPT-4o Vision: Describe Image-Heavy Pages
For pages where text extraction yields little content (charts, diagrams,
dashboards), we call **GPT-4o Vision** to produce a detailed text description.
This description is then embedded and stored in ChromaDB.


In [ ]:
VISION_SYSTEM_PROMPT = """You are a precise document analyst.
Your task: produce a thorough, structured description of the provided page image
so that a text-based retrieval system can accurately find this page when users
ask questions about its content.

Include:
- All numerical values, labels, legends, axis titles visible in charts/graphs
- All text visible in tables, callout boxes, KPI tiles, headers, and footers
- The type of each chart (bar, line, pie, etc.) and what it compares
- Any trends, highlights, or anomalies that are visually apparent
- The page title and section heading if visible

Be comprehensive — a user may ask about any specific number or label on this page.""""

def describe_page_with_vision(page_b64: str, page_num: int) -> str:
    """Call GPT-4o Vision to describe a page image.""""
    response = client.chat.completions.create(
        model=VISION_MODEL,
        max_tokens=1500,
        messages=[
            {"role": "system", "content": VISION_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": f"Describe page {page_num} of this document completely."
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{page_b64}",
                            "detail": "high"
                        }
                    }
                ]
            }
        ]
    )
    return response.choices[0].message.content


# ── Generate descriptions for image-heavy pages ──────────────────
page_descriptions = {}   # page_num → vision description

for tp in text_pages:
    pg = tp["page_num"]
    if tp["char_count"] < MIN_TEXT_CHARS:
        print(f"🔍 Page {pg}: image-heavy ({tp['char_count']} chars) → calling GPT-4o Vision...")
        b64 = page_images[pg - 1]["base64"]
        desc = describe_page_with_vision(b64, pg)
        page_descriptions[pg] = desc
        print(f"   ✅ Description generated ({len(desc):,} chars)")
        # Show a snippet
        print(f"   Preview: {desc[:200]}...\n")
    else:
        print(f"📄 Page {pg}: text-rich ({tp['char_count']:,} chars) → using extracted text")

print("\n✅ Vision descriptions complete")


## 🧩 Cell 8 — Build Unified Chunk List
Merge text pages and vision descriptions into a single list of documents
ready for embedding. Each chunk carries rich metadata.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n\n", "\n", ".", " ", ""]
)

all_chunks = []    # list of {'id', 'text', 'metadata'}
chunk_id   = 0

for tp in text_pages:
    pg   = tp["page_num"]
    src  = Path(PDF_PATH).name

    # ── Decide content: extracted text OR vision description ──────
    if pg in page_descriptions:
        raw_content  = page_descriptions[pg]
        content_type = "image_description"
        label        = f"[Vision Description — Page {pg}]"
    else:
        raw_content  = tp["text"]
        content_type = "text"
        label        = f"[Extracted Text — Page {pg}]"

    if not raw_content.strip():
        print(f"⚠️  Page {pg} has no content, skipping")
        continue

    # Prepend a label so the LLM knows the page number in every chunk
    labelled = f"{label}\n\n{raw_content}"
    sub_chunks = splitter.split_text(labelled)

    for j, chunk_text in enumerate(sub_chunks):
        all_chunks.append({
            "id":       f"page{pg}_chunk{j}",
            "text":     chunk_text,
            "metadata": {
                "source":       src,
                "page_num":     pg,
                "chunk_index":  j,
                "content_type": content_type,   # "text" | "image_description"
                "total_chunks": len(sub_chunks)
            }
        })
        chunk_id += 1

print(f"✅ {len(all_chunks)} total chunks created")
for pg_num in set(c['metadata']['page_num'] for c in all_chunks):
    pg_chunks = [c for c in all_chunks if c['metadata']['page_num'] == pg_num]
    ctype     = pg_chunks[0]['metadata']['content_type']
    print(f"   Page {pg_num}: {len(pg_chunks)} chunks  [{ctype}]")


## 🗄️ Cell 9 — Embed & Store in ChromaDB
We use OpenAI's `text-embedding-3-small` via ChromaDB's built-in
`OpenAIEmbeddingFunction` — no manual embedding loop needed.


In [ ]:
# ── Set up ChromaDB with OpenAI embedding function ────────────────
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name=EMBED_MODEL
)

chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

# Delete existing collection if re-running (idempotent)
try:
    chroma_client.delete_collection(CHROMA_COLLECTION)
    print("🗑️  Deleted existing collection")
except Exception:
    pass

collection = chroma_client.create_collection(
    name=CHROMA_COLLECTION,
    embedding_function=openai_ef,
    metadata={"hnsw:space": "cosine"}   # cosine similarity
)

print(f"✅ ChromaDB collection '{CHROMA_COLLECTION}' created")

# ── Batch upsert all chunks ───────────────────────────────────────
BATCH_SIZE = 50    # ChromaDB handles batches well up to ~100

ids       = [c["id"]       for c in all_chunks]
documents = [c["text"]     for c in all_chunks]
metadatas = [c["metadata"] for c in all_chunks]

for i in range(0, len(ids), BATCH_SIZE):
    collection.upsert(
        ids       = ids[i:i+BATCH_SIZE],
        documents = documents[i:i+BATCH_SIZE],
        metadatas = metadatas[i:i+BATCH_SIZE]
    )
    print(f"   Upserted chunks {i+1}–{min(i+BATCH_SIZE, len(ids))} / {len(ids)}")

print(f"\n✅ {collection.count()} chunks stored in ChromaDB")


## 🔎 Cell 10 — Retrieval Function
Queries ChromaDB and returns the top-k most relevant chunks
along with their metadata and source page numbers.


In [ ]:
def retrieve(query: str, k: int = TOP_K) -> List[Dict]:
    """
    Embed the query and retrieve the top-k matching chunks.
    Returns list of dicts with keys: id, text, metadata, distance
    """
    results = collection.query(
        query_texts=[query],
        n_results=k,
        include=["documents", "metadatas", "distances"]
    )

    chunks = []
    for i in range(len(results["ids"][0])):
        chunks.append({
            "id":       results["ids"][0][i],
            "text":     results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "distance": round(results["distances"][0][i], 4)
        })
    return chunks


# ── Quick test ────────────────────────────────────────────────────
test_query = "What was the total Q4 revenue?"
test_results = retrieve(test_query, k=3)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(test_results)} chunks:\n")
for r in test_results:
    md_info = r['metadata']
    print(f"  [{r['id']}]  Page {md_info['page_num']}  "
          f"type={md_info['content_type']}  dist={r['distance']}")
    print(f"  Preview: {r['text'][:180].strip()}...\n")


## 💬 Cell 11 — Multimodal Answer Generation
When retrieved chunks reference image-description pages, we also attach
the **original page image** to the GPT-4o prompt for maximum grounding.


In [ ]:
# ── Lazy client getter — works even if Cell 2 was skipped ───────────────────
def _get_client():
    """Return the global OpenAI client, initialising it if needed."""
    import openai
    from openai import OpenAI
    global client
    if "client" not in globals() or client is None:
        api_key = os.environ.get("OPENAI_API_KEY", "")
        if not api_key or api_key.startswith("sk-..."):
            raise ValueError(
                "OpenAI API key not set.\n"
                "Run Cell 2 first, or do:\n"
                "  import os; os.environ['OPENAI_API_KEY'] = 'sk-YOUR-KEY'"
            )
        client = OpenAI(api_key=api_key)
        print("🔑 OpenAI client re-initialised from environment variable.")
    return client


SYSTEM_PROMPT = """You are a precise document analyst answering questions
strictly based on the provided context.

Rules:
1. Answer ONLY from the context below. Do not use outside knowledge.
2. Always cite the page number (e.g., "According to Page 2...").
3. When referencing numbers, charts, or visual data, be specific.
4. If the context does not contain the answer, say:
   "The document does not contain information about this."
5. Keep answers clear and concise."""


def build_context_text(chunks):
    parts = []
    for i, c in enumerate(chunks, 1):
        pg    = c["metadata"]["page_num"]
        ctype = c["metadata"]["content_type"]
        parts.append(
            f"--- CONTEXT {i} | Page {pg} | Type: {ctype} ---\n"
            f"{c['text'].strip()}"
        )
    return "\n\n".join(parts)


def get_image_pages_for_context(chunks):
    return list(set(
        c["metadata"]["page_num"]
        for c in chunks
        if c["metadata"]["content_type"] == "image_description"
    ))


def answer_question(query: str, k: int = TOP_K, verbose: bool = True) -> dict:
    """
    Full multimodal RAG pipeline:
      1. Retrieve top-k chunks from ChromaDB
      2. Build text context string
      3. Attach page images for any image-description pages
      4. Call GPT-4o and return a grounded, cited answer
    """
    oai = _get_client()           # ← safe client fetch

    # ── 1. Retrieve ─────────────────────────────────────────────────
    chunks = retrieve(query, k=k)

    if verbose:
        print(f"\n📥 Retrieved {len(chunks)} chunks:")
        for c in chunks:
            print(f"   Page {c['metadata']['page_num']} "
                  f"[{c['metadata']['content_type']}]  dist={c['distance']}")

    # ── 2. Build context text ────────────────────────────────────────
    context_text = build_context_text(chunks)

    # ── 3. Find image pages ──────────────────────────────────────────
    image_page_nums = get_image_pages_for_context(chunks)

    # ── 4. Build message content ─────────────────────────────────────
    user_content = [
        {
            "type": "text",
            "text": (
                f"Context from the document:\n\n"
                f"{context_text}\n\n"
                f"{'--- PAGE IMAGES ATTACHED BELOW ---' if image_page_nums else ''}\n\n"
                f"Question: {query}"
            )
        }
    ]

    for pg_num in sorted(image_page_nums):
        b64 = page_images[pg_num - 1]["base64"]
        user_content.append({"type": "text",
                              "text": f"Image of Page {pg_num}:"})
        user_content.append({
            "type": "image_url",
            "image_url": {"url": f"data:image/png;base64,{b64}",
                          "detail": "high"}
        })
        if verbose:
            print(f"🖼️  Attached Page {pg_num} image to prompt")

    # ── 5. Generate ──────────────────────────────────────────────────
    response = oai.chat.completions.create(
        model=GENERATION_MODEL,
        max_tokens=1200,
        temperature=0,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": user_content}
        ]
    )

    answer = response.choices[0].message.content
    usage  = response.usage

    return {
        "query":            query,
        "answer":           answer,
        "retrieved_chunks": chunks,
        "image_pages_used": image_page_nums,
        "tokens": {
            "prompt":     usage.prompt_tokens,
            "completion": usage.completion_tokens,
            "total":      usage.total_tokens
        }
    }


print("✅ answer_question() ready  (auto-recovers client if session restarted)")


## 🚀 Cell 12 — Ask Questions

> **If you see `NameError: name 'X' is not defined`** — run the guard cell directly below first. It re-initialises all required state without re-processing the PDF.

In [ ]:
# ╔══════════════════════════════════════════════════════════════════╗
# ║  GUARD CELL — run this if you get NameError in Cell 12          ║
# ║  It reloads every dependency without re-processing the PDF.     ║
# ╚══════════════════════════════════════════════════════════════════╝
import os, base64, io, json
from pathlib import Path
from typing import List, Dict
from openai import OpenAI
import chromadb
from chromadb.utils import embedding_functions

# ── Re-apply config (copy from Cell 4 if you changed it) ──────────
TOP_K            = 4
EMBED_MODEL      = "text-embedding-3-small"
VISION_MODEL     = "gpt-4o"
GENERATION_MODEL = "gpt-4o"
CHROMA_COLLECTION = "multimodal_rag"
CHROMA_DIR        = "./chroma_db"

# ── Re-initialise OpenAI client ────────────────────────────────────
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
if not OPENAI_API_KEY or OPENAI_API_KEY == "sk-...":
    OPENAI_API_KEY = input("Paste your OpenAI API key: ").strip()
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
client = OpenAI(api_key=OPENAI_API_KEY)

# ── Re-connect ChromaDB ────────────────────────────────────────────
openai_ef = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY, model_name=EMBED_MODEL
)
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)
collection    = chroma_client.get_collection(
    name=CHROMA_COLLECTION, embedding_function=openai_ef
)
print(f"✅ ChromaDB reconnected  ({collection.count()} chunks)")

# ── Re-load PDF page images (needed for visual grounding) ──────────
from pdf2image import convert_from_path

if "PDF_PATH" not in dir() or not Path(PDF_PATH).exists():
    from google.colab import files
    print("PDF not found in session — please re-upload:")
    uploaded = files.upload()
    PDF_PATH = list(uploaded.keys())[0]

pil_images = convert_from_path(PDF_PATH, dpi=150)
page_images = []
for i, img in enumerate(pil_images):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    b64 = base64.b64encode(buf.getvalue()).decode("utf-8")
    page_images.append({"page_num": i+1, "image": img, "base64": b64, "size": img.size})

print(f"✅ {len(page_images)} page image(s) reloaded")
print("✅ Guard cell complete — re-run Cell 12 now")


In [ ]:
def pretty_print(result: Dict):
    print("=" * 65)
    print(f"❓ QUESTION:\n   {result['query']}")
    print("-" * 65)
    print(f"💬 ANSWER:\n{result['answer']}")
    print("-" * 65)
    print(f"📌 Sources: ", end="")
    source_info = [
        f"Page {c['metadata']['page_num']} [{c['metadata']['content_type']}]"
        for c in result['retrieved_chunks']
    ]
    print(", ".join(source_info))
    if result['image_pages_used']:
        print(f"🖼️  Image pages included in prompt: {result['image_pages_used']}")
    print(f"🔢 Tokens used: {result['tokens']['total']} "
          f"(prompt={result['tokens']['prompt']}, "
          f"completion={result['tokens']['completion']})")
    print("=" * 65)


# ── Question 1: Numerical fact from text ─────────────────────────
r1 = answer_question("What was the total Q4 2024 revenue and how does it compare to Q3?")
pretty_print(r1)


In [ ]:
# ── Question 2: Visual chart data (should pull image page) ────────
r2 = answer_question("Which region closed the most deals and how many? What about LATAM?")
pretty_print(r2)


In [ ]:
# ── Question 3: NPS from dashboard chart ─────────────────────────
r3 = answer_question("What was the NPS score at the end of Q4 and when did it cross the target of 70?")
pretty_print(r3)


In [ ]:
# ── Question 4: Product analysis ─────────────────────────────────
r4 = answer_question("How did SecureVault perform in Q4 and what is its gross margin?")
pretty_print(r4)


In [ ]:
# ── Question 5: Strategic question ───────────────────────────────
r5 = answer_question("What are the Q1 2025 priorities and revenue forecast?")
pretty_print(r5)


## 💡 Cell 13 — Interactive Query Loop
Ask your own questions in a loop. Type `quit` to stop.


In [ ]:
print("🤖 Multimodal RAG — Interactive Mode")
print("   Type your question and press Enter. Type 'quit' to exit.\n")

while True:
    user_q = input("You: ").strip()
    if not user_q or user_q.lower() in ("quit", "exit", "q"):
        print("Goodbye!")
        break
    result = answer_question(user_q, verbose=False)
    print(f"\nBot: {result['answer']}")
    print(f"[Sources: {', '.join(f'Page {c["metadata"]["page_num"]}' for c in result['retrieved_chunks'])}]")
    if result['image_pages_used']:
        print(f"[Image pages: {result['image_pages_used']}]")
    print()


## 🗂️ Cell 14 — Inspect ChromaDB Contents (Optional)
Peek at what's stored in the vector database.


In [ ]:
# View all stored items
all_items = collection.get(include=["documents", "metadatas"])

print(f"Total chunks in ChromaDB: {collection.count()}\n")
print(f"{'ID':<22} {'Page':>4}  {'Type':<20}  {'Chars':>5}  Preview")
print("-" * 80)
for cid, doc, meta in zip(
    all_items["ids"], all_items["documents"], all_items["metadatas"]
):
    preview = doc[:55].replace("\n", " ")
    print(f"{cid:<22} {meta['page_num']:>4}  {meta['content_type']:<20}  "
          f"{len(doc):>5}  {preview}...")


---
## 🏗️ Architecture Summary

```
PDF File
   │
   ├─── pdfplumber ──────────────→ Extracted Text (per page)
   │                                      │
   └─── pdf2image ───────────────→ Page Images (PNG)
              │                           │
              │           text-rich?  ────┤
              │               YES         │ chunk with
              │                           │ RecursiveChar
              │               NO          │ TextSplitter
              └─── GPT-4o Vision ──→ Image Description
                                          │
                          OpenAI text-embedding-3-small
                                          │
                                    ChromaDB (cosine)
                                          │
                              ┌─── Query Embedding
                              │
                          Top-k Retrieve
                              │
                    ┌─────────┴──────────┐
                    │                    │
               Text chunks         Image pages
                    │               (as PNG)
                    └────────┬───────────┘
                             │
                        GPT-4o Prompt
                        (text + images)
                             │
                      Grounded Answer
                      with page citations
```

### Key Design Decisions
| Decision | Rationale |
|----------|-----------|
| Page images always rasterized | Needed for both vision description and GPT-4o visual grounding |
| Vision description only when text < 200 chars | Avoids paying for vision on text-heavy pages |
| Images attached at query time, not index time | Keeps ChromaDB lean (only text embeddings stored) |
| Cosine similarity in ChromaDB | Better for semantic search than L2 on normalized embeddings |
| Chunk label includes page number | Ensures every chunk carries page attribution for citations |
